# Prompt Optimization with Opik: an A-to-Z Guide

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/comet-ml/opik-examples/blob/main/guides/prompt_agent_optimization/prompt_agent_optimization.ipynb)

This notebook takes you from *"I have a prompt that works okay"* to *"my prompt and my agent are measurably better, and every improvement is a comparable run in Opik."*

**This notebook is intended to be used in two ways:**
1. **Live workshop + take-home guide:** run **Part 1** as the workshop element. You'll optimize a RAG answer prompt against an exact-match metric and see the improvement in Opik. The remaining sections can be used for further understanding as in 2.
2. **Self-contained end-to-end guide:** continue through Parts 2–5, LLM-judge metrics (and how to *trust* them), multi-objective optimization, and optimizing an agent's tool use.

We optimize a **documentation assistant** for a fictional product, **Ledgerline** (a task-queue API).

**Runs anywhere.** The notebook installs its own dependencies and configures credentials in the first few cells and defines its tiny RAG app inline, so you can run it top-to-bottom in **Google Colab** (badge above) or locally — nothing else to set up.

📚 **New to optimization in Opik?** See the [Optimization runs overview](https://www.comet.com/docs/opik/development/optimization-runs/overview) in the docs.

## Part 0 — How to think about prompt optimization

**When do you start?** When you have;
1. a prompt that works *okay*, 
2. a dataset of representative inputs, and 
3. a metric that says how good an output is, and 
4. hand-tuning has plateaued.

**The mental shift.** Classic optimization gives you an objective and a gradient. Prompt optimization is different: the **search space is prompt text**, and the **objective is a metric computed over a dataset**. You can't differentiate it, so optimizers *propose* candidate prompts, *evaluate* them on your dataset, keep the best, and repeat.

The three ingredients map exactly to three objects you'll build:

| Ingredient | Opik object |
|---|---|
| The prompt | `ChatPrompt` |
| The dataset | Opik `Dataset` |
| The metric | a callable `(dataset_item, llm_output) -> float` |

The loop, once, looks like: **propose candidate → evaluate on dataset → keep best → repeat.** Everything below is that loop, escalating in complexity.

## Setup

The next few cells make the notebook self-contained: install dependencies, configure your credentials, and define a tiny RAG app over the Ledgerline docs. Run them once, top to bottom.

In [ ]:
# Install dependencies. In Colab this installs them; locally (after `uv sync`)
# they're already present. --upgrade keeps you on current SDKs.
%pip install --quiet --upgrade opik opik-optimizer chromadb litellm ipywidgets

### Credentials

Run the cell below. `opik.configure()` reads your **Opik API key** and **workspace** from the environment, or prompts for them (get them free at [comet.com/opik](https://www.comet.com/opik)).

You also need a **model provider key**: the guide calls models through litellm and defaults to a small Anthropic Claude model, so it'll use (or prompt for) your `ANTHROPIC_API_KEY`. To use another provider, set `OPIK_EXAMPLES_MODEL` (e.g. `openai/gpt-4o-mini`) and it'll use that provider's key instead.

If you need to create a custom model interface, refer to [the OpikBaseModel Interface docs](https://www.comet.com/docs/opik/latest/evaluation/metrics/custom_model#the-opikbasemodel-interface). 

In [ ]:
import getpass
import os

import opik

OPIK_PROJECT_NAME = "prompt-agent-optimization"

# Reads OPIK_API_KEY / OPIK_WORKSPACE from the environment, or prompts for them.
# install_mcp=False keeps it non-interactive so it also runs unattended in CI.
opik.configure(project_name=OPIK_PROJECT_NAME, install_mcp=False)

# The model, called via litellm. Default is a small Anthropic Claude model; set
# OPIK_EXAMPLES_MODEL to switch providers (e.g. "openai/gpt-4o-mini").
MODEL = os.environ.get("OPIK_EXAMPLES_MODEL", "anthropic/claude-haiku-4-5-20251001")

# Its provider key (ANTHROPIC_API_KEY / OPENAI_API_KEY / ...): from env, or prompted.
key_var = f"{MODEL.split('/')[0].upper()}_API_KEY"
if not os.environ.get(key_var):
    os.environ[key_var] = getpass.getpass(f"Enter {key_var}: ")

print("Using model:", MODEL, "| logging to project:", OPIK_PROJECT_NAME)

### The app: a tiny RAG over the Ledgerline docs

Everything the guide needs is defined right here in the notebook — no external files. First the corpus and evaluation cases (for a fictional task-queue product, **Ledgerline**), then a small RAG app: a ChromaDB retriever and an `answer()` function. Swap this corpus for your own product's docs and the rest of the guide still applies.

In [ ]:
# --- Corpus: ~12 short Ledgerline doc snippets. ---
DOCS = [
    {"id": "timeouts", "title": "Job timeouts", "text": "Every Ledgerline job has a default timeout of 30 seconds. Jobs exceeding the timeout are marked failed and eligible for retry. The maximum configurable timeout is 15 minutes."},
    {"id": "retries", "title": "Retries", "text": "Failed jobs are retried automatically. The default maximum number of retries is 3, using exponential backoff starting at 2 seconds. Set max_retries to 0 to disable retries."},
    {"id": "rate-limits", "title": "Rate limits", "text": "The API allows 1000 requests per minute per API key. Exceeding the limit returns HTTP 429. Rate limit headers are included on every response."},
    {"id": "auth", "title": "Authentication", "text": "Authenticate by sending your API key in the Authorization header as a Bearer token: 'Authorization: Bearer <API_KEY>'. Keys are created in the dashboard."},
    {"id": "priorities", "title": "Queue priorities", "text": "Ledgerline supports three queue priorities: low, default, and high. High-priority jobs are dequeued before default and low. Priority is set per job at enqueue time."},
    {"id": "dead-letter", "title": "Dead-letter queue", "text": "After a job exhausts all retries it is moved to the dead-letter queue, where it is retained for 7 days before permanent deletion. Dead-letter jobs can be replayed from the dashboard."},
    {"id": "webhooks", "title": "Webhooks", "text": "When a job completes, Ledgerline POSTs a webhook to your configured URL. The payload includes job_id, status, and result fields. Webhook deliveries are signed with the X-Ledgerline-Signature header."},
    {"id": "install", "title": "SDK installation", "text": "Install the Python SDK with 'pip install ledgerline'. The SDK requires Python 3.9 or newer. Import it as 'import ledgerline'."},
    {"id": "concurrency", "title": "Concurrency", "text": "Each project runs up to 50 concurrent jobs by default. Contact support to raise the concurrency limit for your plan."},
    {"id": "regions", "title": "Regions", "text": "Ledgerline is available in three regions: us-east, eu-west, and ap-south. The default region is us-east. Set the region when initializing the client."},
    {"id": "batch", "title": "Batch enqueue", "text": "You can enqueue up to 500 jobs in a single batch request. Larger batches must be split. Each job in a batch is billed individually."},
    {"id": "idempotency", "title": "Idempotency", "text": "Pass an Idempotency-Key header to safely retry enqueue requests. Ledgerline deduplicates requests with the same key for 24 hours."},
]

# --- Part 1 eval: exact-match cases. Each expected_substring appears verbatim in a doc. ---
EXACT_CASES = [
    {"query": "What is the default job timeout?", "expected_substring": "30 seconds"},
    {"query": "What is the maximum configurable timeout?", "expected_substring": "15 minutes"},
    {"query": "How many times are failed jobs retried by default?", "expected_substring": "3"},
    {"query": "How do I disable retries?", "expected_substring": "max_retries to 0"},
    {"query": "What backoff does retry use, and starting at what delay?", "expected_substring": "2 seconds"},
    {"query": "How many requests per minute per API key are allowed?", "expected_substring": "1000 requests per minute"},
    {"query": "What HTTP status is returned when the rate limit is exceeded?", "expected_substring": "429"},
    {"query": "Which header carries the API key?", "expected_substring": "Authorization"},
    {"query": "What token scheme is used for auth?", "expected_substring": "Bearer"},
    {"query": "What queue priorities are supported?", "expected_substring": "low, default, and high"},
    {"query": "How long are dead-letter jobs retained?", "expected_substring": "7 days"},
    {"query": "Which header signs webhook deliveries?", "expected_substring": "X-Ledgerline-Signature"},
    {"query": "How do I install the Python SDK?", "expected_substring": "pip install ledgerline"},
    {"query": "What Python version does the SDK require?", "expected_substring": "3.9"},
    {"query": "How many concurrent jobs run per project by default?", "expected_substring": "50 concurrent jobs"},
    {"query": "What is the default region?", "expected_substring": "us-east"},
    {"query": "How many jobs can I enqueue in one batch?", "expected_substring": "500 jobs"},
    {"query": "How long are idempotency keys deduplicated?", "expected_substring": "24 hours"},
]

# --- Part 2 eval: open-ended cases with a reference answer (for an LLM judge). ---
JUDGE_CASES = [
    {"query": "How should I handle a job that keeps failing?", "reference": "Explain retries with exponential backoff, the default of 3 retries, and that exhausted jobs move to the dead-letter queue (retained 7 days, replayable from the dashboard)."},
    {"query": "How do I make sure I don't enqueue the same job twice if my request retries?", "reference": "Use an Idempotency-Key header; Ledgerline deduplicates same-key requests for 24 hours."},
    {"query": "What's the best way to authenticate my requests?", "reference": "Send the API key as a Bearer token in the Authorization header; create keys in the dashboard."},
    {"query": "How do I get notified when a job finishes?", "reference": "Configure a webhook URL; Ledgerline POSTs job_id, status, and result, signed with X-Ledgerline-Signature."},
    {"query": "How can I prioritise urgent work?", "reference": "Set the job priority to high at enqueue time; high-priority jobs are dequeued before default and low."},
    {"query": "How do I run more jobs at the same time?", "reference": "Default concurrency is 50 concurrent jobs per project; contact support to raise the limit."},
    {"query": "How do I choose where my jobs run?", "reference": "Set the region (us-east, eu-west, ap-south) when initializing the client; default is us-east."},
    {"query": "What happens when I hit the rate limit?", "reference": "Requests over 1000/min per key return HTTP 429; rate-limit headers are on every response."},
    {"query": "How do I submit many jobs efficiently?", "reference": "Use batch enqueue, up to 500 jobs per request; split larger batches; each job billed individually."},
    {"query": "How long do I have to recover a permanently failing job?", "reference": "Dead-letter jobs are retained 7 days before permanent deletion and can be replayed from the dashboard."},
    {"query": "Can I make jobs run longer than the default?", "reference": "Yes; the default timeout is 30 seconds and the maximum configurable timeout is 15 minutes."},
    {"query": "How do I start using the SDK in Python?", "reference": "Install with pip install ledgerline (Python 3.9+), then import ledgerline."},
]

print(f"{len(DOCS)} docs, {len(EXACT_CASES)} exact cases, {len(JUDGE_CASES)} judge cases")

In [ ]:
import threading

import chromadb
import litellm

client = opik.Opik()

# --- A tiny RAG app: an in-memory ChromaDB retriever + an answer function. ---
_collection = chromadb.Client().get_or_create_collection(
    "ledgerline_docs", metadata={"hnsw:space": "cosine"}
)
# The optimizer calls retrieve() across worker threads; serialize reads for safety.
_retrieve_lock = threading.Lock()


def ingest(docs):
    _collection.upsert(
        ids=[d["id"] for d in docs],
        documents=[d["text"] for d in docs],
        metadatas=[{"title": d["title"]} for d in docs],
    )
    return _collection.count()

@opik.track(project_name=OPIK_PROJECT_NAME)
def retrieve(query, n_results=3):
    with _retrieve_lock:
        result = _collection.query(query_texts=[query], n_results=n_results)
    return result["documents"][0]


@opik.track(project_name=OPIK_PROJECT_NAME)
def answer(query, system_prompt, model=None):
    context = "\n\n".join(retrieve(query))
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ]
    response = litellm.completion(model=model or MODEL, messages=messages)
    return response.choices[0].message.content


def build_dataset(name, cases):
    dataset = client.get_or_create_dataset(name)
    dataset.insert(cases)
    return dataset


def optimized_system(result):
    """The optimized system text. An optimizer-returned ChatPrompt stores its text
    in messages (get_messages()), not the .system scalar (which stays None)."""
    for message in result.prompt.get_messages():
        if message["role"] == "system":
            return message["content"]
    return None

## Part 1: Your first optimization

We'll ingest the Ledgerline docs, build an evaluation dataset with **checkable answers**, score a baseline prompt with an **exact-match metric** (no LLM judge needed), then let an optimizer improve the prompt.

In [ ]:
count = ingest(DOCS)
print(f"Ingested {count} doc snippets into ChromaDB")

In [ ]:
# RAG step — retrieve. For each question, pull the most relevant docs with our
# retriever and attach them as `context`. A production RAG system retrieves per
# query at answer time; we do it once here so every optimizer trial answers the
# SAME question from the SAME context. What we optimize is the *prompt*, not the
# retriever.
for case in EXACT_CASES:
    case["context"] = "\n\n".join(retrieve(case["query"]))

exact_dataset = build_dataset("ledgerline-exact", EXACT_CASES)
print(f"Dataset 'ledgerline-exact' has {len(EXACT_CASES)} cases (each with retrieved context)")

In [ ]:
import json

# Peek at one dataset item. The optimizer fills {query}/{context} in the prompt
# from these fields; the exact-match metric checks that expected_substring appears.
print(json.dumps(EXACT_CASES[0], indent=2))

A dataset item looks like this — the question, the fact we check for, and the docs our retriever pulled for it (context abridged; the cell above prints it in full):

```json
{
  "query": "What is the default job timeout?",
  "expected_substring": "30 seconds",
  "context": "Every Ledgerline job has a default timeout of 30 seconds. ... The maximum configurable timeout is 15 minutes.\n\nFailed jobs are retried automatically. The default maximum number of retries is 3 ...\n\nAfter a job exhausts all retries it is moved to the dead-letter queue, ... retained for 7 days ..."
}
```

`{query}` and `{context}` are filled into the prompt from these fields; the metric checks that `expected_substring` shows up in the answer.

#### The metric: exact-match, no judge

Our first metric is deterministic and cheap: **does the answer contain the expected fact?** Opik ships `Contains` for exactly this. Optimizer metrics are plain callables `(dataset_item, llm_output) -> float`, so we wrap `Contains` in one. *Not every metric needs an LLM.*

In [ ]:
from opik.evaluation.metrics import Contains


def exact_match(dataset_item: dict, llm_output: str) -> float:
    # Contains returns 1.0 if expected_substring is in the output, else 0.0.
    result = Contains(case_sensitive=False).score(
        output=llm_output,
        reference=dataset_item["expected_substring"],
    )
    return result.value


exact_match.__name__ = "exact_match"

#### The starting prompt

Our baseline is a **generic assistant**, `"You are a helpful assistant."`, with no instruction to use the context or be precise. It's deliberately naive, so there's room to improve.

The **user template** injects the retrieved docs as `{context}` and the question as `{query}` — the **retrieve-then-generate** shape of a real RAG system. Retrieval is held fixed; what we optimize is how the **system prompt** turns that context into a good answer.

In [ ]:
from opik_optimizer import ChatPrompt

BASELINE_SYSTEM = "You are a helpful assistant."

prompt = ChatPrompt(
    name="ledgerline-answer",
    system=BASELINE_SYSTEM,
    # user="Context:\n{context}\n\nQuestion: {query}",
    user="Question: {query}",
    model=MODEL,
)

#### Run the optimizer

We use **`MetaPromptOptimizer`**, it uses a reasoning LLM to critique and rewrite the prompt. It's a solid general-purpose starting point for prompt wording. Watch the params:
- `max_trials` — how many candidate prompts to try.
- `n_samples` — dataset rows evaluated per candidate (smaller = cheaper/faster for a live run).
- `skip_perfect_score=False` — keep optimizing even if the baseline already scores high.

In [ ]:
from opik_optimizer import MetaPromptOptimizer

optimizer = MetaPromptOptimizer(
    model=MODEL,
    n_threads=4,
    skip_perfect_score=False,
)

result = optimizer.optimize_prompt(
    prompt=prompt,
    dataset=exact_dataset,
    metric=exact_match,
    max_trials=8,
    n_samples=8,
)

print("Baseline score:", result.initial_score)
print("Best score:    ", result.score)

# See HOW the prompt was refined: the optimizer rewrote the *system* instructions.
print("\n--- Baseline system prompt ---")
print(BASELINE_SYSTEM)
print("\n--- Optimized system prompt ---")
print(optimized_system(result))

#### What actually changed: baseline vs optimized *answer*

The scores moved, but the point lands when you read the answers. Below, a **trickier, open-ended question** runs through the **naive baseline** prompt and the **optimized** prompt (via the real `answer()` function, which retrieves then generates). Watch the baseline sprawl while the optimized answer stays concise and grounded.

In [ ]:
# A trickier, open-ended question shows the contrast best: the naive baseline
# sprawls, the optimized prompt stays concise and grounded.
q = "How should I handle a job that keeps failing?"
print("Question:", q)
print("\n--- BASELINE answer ---")
print(answer(q, system_prompt=BASELINE_SYSTEM))
print("\n--- OPTIMIZED answer ---")
print(answer(q, system_prompt=optimized_system(result)))

#### See it in Opik

The cell above printed the **baseline vs optimized system prompt** side by side — that rewrite is the concrete refinement the optimizer found. Now open **Evaluation → Optimization runs** in your Opik workspace: you'll see this run with every candidate prompt, its score, and the trace for each trial. Compare the baseline row to the best row — that delta is your improvement.

## Part 2: Metrics done right

Exact-match got us far because our questions had crisp answers. But real docs questions are open-ended, *"How should I handle a job that keeps failing?"* has no single substring. For those you need a metric that judges **meaning**: an **LLM-as-judge**.

#### The LLM-judge metric

Open-ended questions have no single substring to match, so we score **meaning** with an LLM judge. Opik ships judge metrics like `AnswerRelevance` (is the answer relevant to the question, given context?) and `Hallucination` (is it unsupported by context?).

There's a bit more **setup** than `Contains`: a judge needs its own **model** to do the scoring, and it takes the question, the answer, and a reference/context, not just the output. But the wrapper is the same shape: a callable `(dataset_item, llm_output) -> float`.

In [ ]:
from opik.evaluation.metrics import AnswerRelevance


def answer_relevance(dataset_item: dict, llm_output: str) -> float:
    result = AnswerRelevance(model=MODEL).score(
        input=dataset_item["query"],
        output=llm_output,
        context=[dataset_item["reference"]],
    )
    return result.value


answer_relevance.__name__ = "answer_relevance"

#### How do we *trust* a judge?

An LLM-judge is itself a prompt — it can be wrong. Before you optimize *against* it, sanity-check it:

1. **Spot-check against your own labels.** Take 3–5 rows, decide the score yourself, and compare. If you and the judge disagree wildly, fix the judge before trusting its numbers.
2. **Read the *reason*, not just the number.** Opik judge metrics return a `reason`. A right score for the wrong reason is a red flag.
3. **Watch for drift and bias.** Judges favor longer, confident-sounding answers. If your metric rewards verbosity, your "optimized" prompt may just be wordier — which is exactly why Part 2 ends with a *cost* objective.

4. **Mind the judge's model.** Here the judge runs on the *same* model that wrote the answers — fine for a demo, but in production prefer a separate (often stronger) judge so a model isn't grading its own work.

Run the cell below to inspect a judge score **and its reasoning** on one example.

In [ ]:
sample = JUDGE_CASES[0]
sample_output = answer(sample["query"], system_prompt=optimized_system(result))
judged = AnswerRelevance(model=MODEL).score(
    input=sample["query"],
    output=sample_output,
    context=[sample["reference"]],
)
print("Question:", sample["query"])
print("Answer:  ", sample_output)
print("Score:   ", judged.value)
print("Reason:  ", judged.reason)

#### Run the optimizer against the judge

Now the key point: **it's the same loop as Part 1.** Same `MetaPromptOptimizer`, same `optimize_prompt` call — we only swap the *metric* from `exact_match` to `answer_relevance`. Optimizer and metric are independent choices; you mix and match.

First build the eval dataset (open-ended questions need a `reference` answer, which the exact-match set didn't), then run it.

In [ ]:
# Judge dataset: open-ended questions, each with a reference answer. We attach the
# retrieved context the same way as Part 1.
for case in JUDGE_CASES:
    case["context"] = "\n\n".join(retrieve(case["query"]))
judge_dataset = build_dataset("ledgerline-judge", JUDGE_CASES)
print(f"Dataset 'ledgerline-judge' has {len(JUDGE_CASES)} open-ended cases")

In [ ]:
# Same optimizer as Part 1 (reused) — only the metric changed.
judge_prompt = ChatPrompt(
    name="ledgerline-answer-judge",
    system=BASELINE_SYSTEM,
    user="Context:\n{context}\n\nQuestion: {query}",
    model=MODEL,
)

judge_result = optimizer.optimize_prompt(
    prompt=judge_prompt,
    dataset=judge_dataset,
    metric=answer_relevance,
    max_trials=8,
    n_samples=8,
)
print("Judge-metric baseline:", judge_result.initial_score, "-> best:", judge_result.score)

#### Multi-objective: quality *and* cost

Optimizing purely for a judge can inflate answer length and **cost**. Often you want **quality high *and* cost low**. `MultiMetricObjective` combines metrics into one weighted composite the optimizer maximizes.

For cost we use Opik's built-in **`SpanCost`** metric: it reads the **actual token usage** from each answer's trace and prices it (via litellm's up-to-date pricing tables), normalized against a soft per-answer budget (`target`) so cheaper answers score higher. We weight quality and cost equally (0.5 / 0.5).

In [ ]:
from opik_optimizer import MultiMetricObjective
from opik_optimizer.metrics import SpanCost

# Real cost, not a length proxy: SpanCost reads each answer's actual token usage from
# its trace and prices it via litellm, normalized against a soft per-answer budget so
# cheaper -> higher score. This is the built-in, recommended way to add cost to a
# MultiMetricObjective.
cost = SpanCost(target=0.001)  # target USD per answer; tune to your model + budget

composite = MultiMetricObjective(
    metrics=[answer_relevance, cost],
    weights=[0.5, 0.5],
    name="relevance_and_cost",
)

multi_result = optimizer.optimize_prompt(
    prompt=judge_prompt,
    dataset=judge_dataset,
    metric=composite,
    max_trials=8,
    n_samples=8,
)
print("Multi-objective best score:", multi_result.score)
print("\nOptimized system prompt:\n", optimized_system(multi_result))

#### Compare your runs

You now have three optimization runs in Opik: exact-match, judge, and multi-objective. In **Evaluation → Optimization runs**, put them side by side. Notice how the multi-objective prompt trades a little relevance for noticeably cheaper answers, that trade-off is the whole point of naming your objectives explicitly.

## Part 3: From prompt to agent

So far retrieval was **fixed**: we retrieved once, put the docs in the prompt, and optimized the wording. Real systems are agents — they *decide* what to do. Here we hand the model a **`search_docs` tool** wired to our retriever and let it choose when to call it. That turns the prompt into an **agent**, and the same optimizer loop tunes it.

**What "optimizing an agent" means:** not rewriting the tool's code — the retriever is fixed. It means optimizing the natural-language surface the agent reasons over: its **system prompt** (when to search, how to answer from results) and, optionally, its **tool descriptions** (`optimize_tools=True`) so it calls the tool at the right moments.

*(A lighter alternative to a tool is a yes/no retrieval gate — a small prompt that decides whether to look up docs at all — optimized with the same loop. We use a real tool here because it better reflects a production agent.)*

In [ ]:
# The tool the agent may call. It wraps our retriever; the *agent* decides when to
# call it. Returning one string keeps the tool result clean.
def search_docs(query: str) -> str:
    """Search the Ledgerline documentation and return the most relevant snippets."""
    return "\n\n".join(retrieve(query))


SEARCH_DOCS_TOOL = {
    "type": "function",
    "function": {
        "name": "search_docs",
        "description": "Search the Ledgerline product documentation for relevant snippets.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "What to look up in the docs."},
            },
            "required": ["query"],
        },
    },
}

AGENT_SYSTEM = "You are a Ledgerline support agent. Use tools when they help."

# tools + function_map make this ChatPrompt an agent: on a tool call, the optimizer
# executes search_docs and feeds the result back to the model.
agent_prompt = ChatPrompt(
    name="ledgerline-agent",
    system=AGENT_SYSTEM,
    user="{query}",
    tools=[SEARCH_DOCS_TOOL],
    function_map={"search_docs": search_docs},
    model=MODEL,
)

# optimize_prompts defaults to "system": we tune the agent's instructions.
# (Flip optimize_tools=True to ALSO let the optimizer refine the tool description.)
agent_result = optimizer.optimize_prompt(
    prompt=agent_prompt,
    dataset=judge_dataset,
    metric=composite,
    max_trials=8,
    n_samples=8,
    allow_tool_use=True,
)
print("Agent baseline:", agent_result.initial_score, "-> best:", agent_result.score)
print("\n--- Optimized agent system prompt ---")
print(optimized_system(agent_result))

#### When demonstrations matter: Few-Shot Bayesian

If the win comes from *showing examples* rather than rewording instructions, reach for `FewShotBayesianOptimizer`, it uses Bayesian search (Optuna) to pick the best set and order of few-shot demonstrations to attach. We point it at the **same agent**, so it tunes the agent's examples rather than a fresh prompt.

In [ ]:
from opik_optimizer import FewShotBayesianOptimizer

fewshot_optimizer = FewShotBayesianOptimizer(model=MODEL, n_threads=4)

fewshot_result = fewshot_optimizer.optimize_prompt(
    prompt=agent_prompt,
    dataset=judge_dataset,
    metric=answer_relevance,
    n_samples=8,
)
print("Few-shot best score:", fewshot_result.score)

#### Tuning the model, not the prompt: Parameter optimizer

Sometimes the prompt is fine and you just need better sampling settings. `ParameterOptimizer` leaves the prompt alone and searches temperature / top_p with Bayesian optimization. It's the right reach when behavior, not wording, is the problem. See the [Parameter optimizer docs](https://www.comet.com/docs/opik/agent_optimization/algorithms/parameter_optimizer) for the search-space API.

## Part 4: Choosing an optimizer

You've now *used* several optimizers at the moment each was the right tool. Here's the consolidated map:

| Optimizer | Best for | You saw it in |
|---|---|---|
| **MetaPrompt** | General prompt rewording & clarity | Part 1 |
| **HRPO** | Systematic fixes from *why* prompts fail (failure-mode analysis) | (try on your own) |
| **Few-Shot Bayesian** | Picking the best demonstrations | Part 3 |
| **Evolutionary** | Exploring diverse structures; multi-objective | (try on your own) |
| **GEPA** | Single-turn, reflection-heavy tasks (`pip install gepa`) | (try on your own) |
| **Parameter** | Temperature / top_p, prompt unchanged | Part 3 (described) |

**How to choose, in four questions:**
1. **What's the constraint** — wording, examples, tool use, or sampling params?
2. **Is the dataset ready** — reflective optimizers (HRPO) need metrics with detailed *reasons*. Split train/validation to avoid overfitting.
3. **What's the budget** — Evolutionary/GEPA burn more tokens than MetaPrompt.
4. **Can you chain?** — e.g. MetaPrompt to fix wording, then Parameter to tune sampling.

The docs' own advice: **start with GEPA or HRPO** for a new task, then specialize.

#### Chaining optimizers

Because every optimizer shares the same API and returns an `OptimizationResult` whose `.prompt` you can feed into the next, you can chain them: optimize wording, then feed the winner into a Parameter run. See [Chaining optimizers](https://www.comet.com/docs/opik/agent_optimization/advanced/chaining_optimizers).

## Part 5: Take it further

**Version the winner.** Promote your best prompt to the Opik **Prompt Library** so it's versioned and reusable.

In [ ]:
best_prompt = client.create_prompt(name="ledgerline-answer", prompt=optimized_system(multi_result))
print("Saved prompt version:", best_prompt.commit)

**Where to go next:**
- **[Optimization Studio](https://www.comet.com/docs/opik/agent_optimization/optimization_studio)** — run all of this from the Opik UI, no code.
- **[Optimizer benchmarks](https://www.comet.com/docs/opik/agent_optimization/algorithms/benchmarks)** — numbers per algorithm.
- **[Agent optimization overview](https://www.comet.com/docs/opik/agent_optimization/overview)** — the full reference.
- **Wrap this in a CLI** — the retriever + `answer()` helpers are defined inline above; lifting them into a module to back a repeatable CLI is a natural next project (out of scope here).

You've gone A-to-Z: framing → first optimization → trustworthy judge metrics → multi-objective → agent tuning → optimizer selection → versioned prompt. Every step is a comparable run in Opik.